<a href="https://colab.research.google.com/github/Deanx270/green-screen-sprite-cleaner/blob/main/green_screen_sprite_cleaner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chroma-Key Sprite Cleaner (Green / Magenta / Blue / Custom) — v1.1
This version fixes artificial green fringing produced by the magenta despill mode. It also clears RGB color data outside transparent pixels (with a small 2-pixel edge bleed for scaling).

**Usage**: Run the cell below, upload your original chroma-key frames, choose the key color, preview and process to ZIP. Use the original frames, not the previously exported PNGs.

In [ ]:
# ONE-CELL GOOGLE COLAB CHROMA-KEY SPRITE CLEANER | MAGENTA GREEN-FRINGE FIX
import io
import zipfile
from pathlib import Path

import cv2
import numpy as np
import ipywidgets as widgets
from PIL import Image, ImageDraw
from scipy.ndimage import distance_transform_edt
from google.colab import files, output as colab_output
from IPython.display import display, clear_output, FileLink

colab_output.enable_custom_widget_manager()


def remove_chroma_screen(image, background='auto', custom_color='#ff00ff',
                         edge_trim=0.5, despill=0.90,
                         color_repair=0.90, tolerance=10,
                         softness=0.45, clean_depth=5):
    rgb = np.asarray(image.convert('RGB'), dtype=np.float32)
    h, w = rgb.shape[:2]

    # Sample the ACTUAL background color from the four corners; the menu
    # selects which chroma family to remove, not an assumed perfect RGB value.
    p = max(5, min(32, h // 12, w // 12))
    corners = np.concatenate([
        rgb[:p, :p].reshape(-1, 3),
        rgb[:p, -p:].reshape(-1, 3),
        rgb[-p:, :p].reshape(-1, 3),
        rgb[-p:, -p:].reshape(-1, 3)
    ])
    screen = np.median(corners, axis=0)

    # These chroma scores distinguish magenta from red/orange character parts.
    # For instance, pure red is NOT treated as magenta.
    def chroma_score(pixels, kind):
        r, g, b = pixels[..., 0], pixels[..., 1], pixels[..., 2]
        if kind == 'green':
            return g - np.maximum(r, b)
        if kind == 'magenta':
            return np.minimum(r, b) - g
        if kind == 'blue':
            return b - np.maximum(r, g)
        raise ValueError('Unknown preset background.')

    chosen = background.lower()
    if chosen == 'auto':
        candidates = ('green', 'magenta', 'blue')
        chosen = max(candidates, key=lambda mode: float(chroma_score(screen, mode)))

    if chosen == 'custom':
        hex_code = custom_color.lstrip('#')
        if len(hex_code) != 6:
            raise ValueError('Custom key color must be a 6-digit HEX color.')
        picked = np.array([int(hex_code[i:i+2], 16) for i in (0, 2, 4)],
                          dtype=np.float32)
        # The picker controls chroma direction; sampled corners still supply
        # the real screen's luminance and actual spill color.
        axis = picked - np.mean(picked)
        size = float(np.linalg.norm(axis))
        if size < 25:
            raise ValueError('Select a saturated custom color, not gray/white/black.')
        axis /= size
        actual_axis = screen - np.mean(screen)
        cosine = float(np.dot(actual_axis, axis) /
                       (np.linalg.norm(actual_axis) + 1e-6))
        if cosine < 0.50:
            raise ValueError('Custom color does not match the frame corners. '
                             'Pick a closer background color.')
        score = np.tensordot(rgb, axis, axes=([2], [0]))
        screen_score = float(np.dot(screen, axis))
    else:
        score = chroma_score(rgb, chosen)
        screen_score = float(chroma_score(screen, chosen))

    if screen_score < 25:
        raise ValueError(
            f'Corners do not look like a {chosen} screen. '
            'Change Background to remove (or try Auto Detect).'
        )

    # Approximate subject coverage for finding its crop and clean interior.
    rough = np.clip(1 - score / screen_score, 0, 1)
    difference = np.max(np.abs(rgb - screen), axis=2)
    extent = (rough > 0.06) & (difference > tolerance)

    if not np.any(extent):
        return Image.new('RGBA', (w, h), (0, 0, 0, 0))

    yy, xx = np.where(extent)
    pad = 22
    y0, y1 = max(0, int(yy.min()) - pad), min(h, int(yy.max()) + pad + 1)
    x0, x1 = max(0, int(xx.min()) - pad), min(w, int(xx.max()) + pad + 1)
    c = rgb[y0:y1, x0:x1]
    rough = rough[y0:y1, x0:x1]
    difference = difference[y0:y1, x0:x1]

    # Find clean inner pixels, not green-tainted colors right at the edges.
    core = (rough > 0.43) & (difference > 28)
    kernel_size = 2 * int(clean_depth) + 1
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,
                                      (kernel_size, kernel_size))
    core = cv2.erode(core.astype(np.uint8), kernel) > 0
    if not np.any(core):
        core = (rough > 0.75) & (difference > 35)
    if not np.any(core):
        return Image.new('RGBA', (w, h), (0, 0, 0, 0))

    # Reconstruct what each edge pixel's color and alpha should have been.
    _, nearest = distance_transform_edt(~core, return_indices=True)
    clean_color = c[nearest[0], nearest[1]]
    direction = clean_color - screen
    delta = c - screen
    alpha_raw = np.clip(
        np.sum(delta * direction, axis=2) /
        (np.sum(direction * direction, axis=2) + 1e-5), 0, 1
    )
    alpha_raw = np.where(difference < tolerance, 0, alpha_raw)
    alpha = np.clip((alpha_raw - 0.008) / 0.992, 0, 1)

    if softness > 0:
        blurred = cv2.GaussianBlur(alpha, (0, 0), 0.65)
        alpha = (1 - softness) * alpha + softness * blurred

    # Pixel-based erosion is separate from color cleanup.
    if edge_trim > 0:
        small_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
        whole = int(edge_trim)
        fraction = float(edge_trim - whole)
        for _ in range(whole):
            alpha = cv2.erode(alpha, small_kernel)
        if fraction > 0:
            alpha = ((1 - fraction) * alpha +
                     fraction * cv2.erode(alpha, small_kernel))

    alpha = np.clip(alpha, 0, 1)
    alpha[alpha < 0.012] = 0
    alpha[alpha > 0.992] = 1

    # Use pre-erosion coverage for color reconstruction. Otherwise, trimming
    # the silhouette would artificially brighten/darken the edge colors.
    coverage = np.maximum(alpha_raw, 0.05)
    foreground = np.clip(
        (c - (1 - coverage[..., None]) * screen) / coverage[..., None],
        0, 255
    )

    visible = (alpha > 0.01).astype(np.uint8)
    inner_distance = cv2.distanceTransform(visible, cv2.DIST_L2, 3)
    edge_zone = np.clip((3.3 - inner_distance) / 3.3, 0, 1)

    # Despill the ACTUAL keyed color channels; not green-only.
    # We also retain the clean interior color to protect naturally saturated
    # subject regions (such as an orange fire tail against magenta).
    if chosen == 'green':
        green_excess = np.maximum(
            foreground[..., 1] -
            np.maximum(foreground[..., 0], foreground[..., 2]) - 3, 0
        )
        foreground[..., 1] -= (
            green_excess * despill * (0.2 + 0.8 * edge_zone)
        )
    elif chosen == 'magenta':
        # Never subtract red AND blue from a magenta edge. Doing so can
        # create a green cast, even if the original frame contains no green.
        # Instead, blend only excess magenta toward the nearest clean sprite
        # color; the alpha unmixing above has already removed most spill.
        excess_magenta = np.maximum(
            np.minimum(foreground[..., 0], foreground[..., 2]) - foreground[..., 1]
            - (np.minimum(clean_color[..., 0], clean_color[..., 2]) - clean_color[..., 1])
            - 5, 0
        )
        weight = np.clip(excess_magenta / 50, 0, 1) * despill * edge_zone * 0.60
        foreground = (foreground * (1 - weight[..., None]) +
                      clean_color * weight[..., None])
    elif chosen == 'blue':
        blue_excess = np.maximum(
            foreground[..., 2] -
            np.maximum(foreground[..., 0], foreground[..., 1]) - 3, 0
        )
        foreground[..., 2] -= (
            blue_excess * despill * (0.2 + 0.8 * edge_zone)
        )
    else:
        # For arbitrary custom colors, remove residual chroma in the selected
        # hue direction only where it exceeds the nearby inner subject color.
        edge_bias = foreground - clean_color
        magnitude = np.maximum(np.sum(edge_bias * axis, axis=2), 0)
        foreground -= (magnitude * despill * (0.2 + 0.8 * edge_zone))[..., None] * axis

    # Replace remaining stained edge colors with nearby CLEAN inner colors.
    repair = np.clip(
        color_repair * (1.2 * edge_zone + 0.25 * (1 - alpha_raw)), 0, 1
    )
    foreground = (foreground * (1 - repair[..., None]) +
                  clean_color * repair[..., None])

    # Prevent artificial green fringing introduced by chroma recovery.
    # A true green highlight remains safe if it already occurs in the source
    # or in the nearby clean sprite interior. The check acts on visible edges.
    if chosen == 'magenta':
        source_green = np.maximum(c[..., 1] - np.maximum(c[..., 0], c[..., 2]), 0)
        clean_green = np.maximum(clean_color[..., 1] -
                                 np.maximum(clean_color[..., 0], clean_color[..., 2]), 0)
        allowed = np.maximum(source_green, clean_green) + 2
        created_green = np.maximum(
            foreground[..., 1] - np.maximum(foreground[..., 0], foreground[..., 2])
            - allowed, 0
        )
        foreground[..., 1] -= created_green

    # Store RGB bleed only 2 px outside the alpha silhouette, not across
    # the entire transparent canvas. This is useful for scaled game sprites
    # without retaining ghost images in completely invisible RGB channels.
    visible = alpha > (0.5 / 255)
    bleed_pixels = distance_transform_edt(~visible) <= 2
    keep_color = visible | bleed_pixels
    foreground = np.where(keep_color[..., None], foreground, 0)

    rgba = np.zeros((h, w, 4), dtype=np.uint8)
    rgba[y0:y1, x0:x1, :3] = np.clip(foreground, 0, 255).astype(np.uint8)
    rgba[y0:y1, x0:x1, 3] = np.clip(alpha * 255, 0, 255).astype(np.uint8)
    return Image.fromarray(rgba, 'RGBA')


def current_settings():
    return dict(
        background=background_selector.value,
        custom_color=custom_picker.value,
        edge_trim=trim_slider.value,
        despill=despill_slider.value / 100,
        color_repair=repair_slider.value / 100,
        tolerance=tolerance_slider.value,
        softness=soft_slider.value / 100,
        clean_depth=depth_slider.value
    )


def preview_panels(rgba):
    bbox = rgba.getchannel('A').getbbox()
    if not bbox:
        return Image.new('RGB', (500, 200), (40, 40, 40))
    margin = 26
    x0, y0, x1, y1 = bbox
    crop = rgba.crop((max(0, x0-margin), max(0, y0-margin),
                      min(rgba.width, x1+margin), min(rgba.height, y1+margin)))
    crop.thumbnail((430, 550), Image.Resampling.LANCZOS)
    gap, head = 14, 28
    canvas = Image.new('RGB', (2 * crop.width + gap, crop.height + head), '#232323')
    for i, (name, color) in enumerate([
        ('DARK PREVIEW', (40, 40, 40, 255)),
        ('PURPLE PREVIEW', (98, 18, 94, 255))
    ]):
        base = Image.new('RGBA', crop.size, color)
        base.alpha_composite(crop)
        canvas.paste(base.convert('RGB'), (i * (crop.width + gap), head))
        ImageDraw.Draw(canvas).text((i * (crop.width + gap) + 5, 6), name,
                                   fill='white')
    return canvas


print('Upload ORIGINAL green/magenta/blue-screen PNG/JPG/WEBP frames (multiple files allowed).')
uploaded = files.upload()
images = {
    name: data for name, data in uploaded.items()
    if Path(name).suffix.lower() in ('.png', '.jpg', '.jpeg', '.webp')
}
if not images:
    raise ValueError('No supported images uploaded.')

slider_layout = widgets.Layout(width='650px')
background_selector = widgets.Dropdown(
    options=[('Auto Detect (from corners)', 'auto'),
             ('Green', 'green'), ('Magenta / Pink', 'magenta'),
             ('Blue', 'blue'), ('Custom (pick a color)', 'custom')],
    value='auto', description='Background to remove',
    style={'description_width': '175px'}, layout=slider_layout)
custom_picker = widgets.ColorPicker(
    concise=False, value='#ff00ff', description='Custom background',
    style={'description_width': '175px'},
    layout=widgets.Layout(width='390px', display='none'))

def toggle_custom_picker(change):
    custom_picker.layout.display = ('flex' if change['new'] == 'custom' else 'none')

background_selector.observe(toggle_custom_picker, names='value')
slider_style = {'description_width': '175px'}
trim_slider = widgets.FloatSlider(value=0.5, min=0, max=2.5, step=0.25,
    description='Edge trim (pixels)', style=slider_style, layout=slider_layout,
    continuous_update=False)
despill_slider = widgets.IntSlider(value=90, min=0, max=100, step=5,
    description='Chroma despill (%)', style=slider_style, layout=slider_layout,
    continuous_update=False)
repair_slider = widgets.IntSlider(value=90, min=0, max=100, step=5,
    description='Edge color repair (%)', style=slider_style, layout=slider_layout,
    continuous_update=False)
tolerance_slider = widgets.IntSlider(value=10, min=4, max=28, step=2,
    description='Background tolerance', style=slider_style, layout=slider_layout,
    continuous_update=False)
soft_slider = widgets.IntSlider(value=45, min=0, max=100, step=5,
    description='Edge smoothing (%)', style=slider_style, layout=slider_layout,
    continuous_update=False)
depth_slider = widgets.IntSlider(value=5, min=2, max=8, step=1,
    description='Clean color depth (px)', style=slider_style, layout=slider_layout,
    continuous_update=False)
file_selector = widgets.Dropdown(options=list(images),
    description='Preview image', style=slider_style, layout=slider_layout)
preview_btn = widgets.Button(description='Update preview', button_style='info',
                             icon='eye')
export_btn = widgets.Button(description='Process all + download ZIP',
                            button_style='success', icon='download')
progress = widgets.IntProgress(min=0, max=len(images), value=0,
                               description='Batch', layout=slider_layout)
preview_output = widgets.Output()
status_output = widgets.Output()


def on_preview(_):
    preview_btn.disabled = True
    with preview_output:
        clear_output(wait=True)
        try:
            name = file_selector.value
            im = Image.open(io.BytesIO(images[name]))
            result = remove_chroma_screen(im, **current_settings())
            display(preview_panels(result))
            print(f'Preview: {name} | Key: {background_selector.value} | Original canvas: {result.width} × {result.height}')
        except Exception as exc:
            print(f'Preview failed: {exc}')
        finally:
            preview_btn.disabled = False


def on_export(_):
    export_btn.disabled = True
    preview_btn.disabled = True
    progress.value = 0
    done = 0
    output_zip = '/content/clean_game_sprites.zip'
    with status_output:
        clear_output(wait=True)
        try:
            settings = current_settings()
            with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
                for name, data in images.items():
                    try:
                        im = Image.open(io.BytesIO(data))
                        result = remove_chroma_screen(im, **settings)
                        png = io.BytesIO()
                        result.save(png, format='PNG')
                        zf.writestr(Path(name).stem + '_transparent.png',
                                    png.getvalue())
                        done += 1
                        print(f'✓ {name}')
                    except Exception as exc:
                        print(f'✗ {name}: {exc}')
                    progress.value += 1
            if done:
                print(f'Completed {done}/{len(images)}. Starting ZIP download...')
                display(FileLink(output_zip))
                files.download(output_zip)
            else:
                print('No files could be processed.')
        except Exception as exc:
            print(f'Export failed: {exc}')
        finally:
            export_btn.disabled = False
            preview_btn.disabled = False


preview_btn.on_click(on_preview)
export_btn.on_click(on_export)
display(widgets.VBox([
    widgets.HTML('<h3>Chroma-Key Sprite Cleaner</h3>'
                 '<div>Choose the background color, adjust sliders, preview, then export.</div>'),
    file_selector, background_selector, custom_picker,
    trim_slider, despill_slider, repair_slider,
    tolerance_slider, soft_slider, depth_slider,
    widgets.HBox([preview_btn, export_btn]),
    progress, preview_output, status_output
]))
on_preview(None)